# 📱 Impact of Social Media on Student Life
**Dataset:** Social_media_impact_on_life.csv (4,500 students)  
**Columns:** Age, Gender, Academic Level, Primary Platform, Daily Usage Hours, Sleep Duration, Stress Score, Mental Health Index, GPA, Overall Impact, and more.

### Notebook Structure
1. Environment Setup & Data Loading
2. Data Cleaning & Quality Checks
3. Exploratory Data Analysis (EDA)
4. Visualizations
5. Correlation & Statistical Analysis
6. Machine Learning — Classification (Overall Impact)
7. Machine Learning — Regression (GPA Prediction)
8. Key Findings & Conclusions

---
## 1. Environment Setup & Data Loading

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

# ── Statistical libraries ────────────────────────────────────────────────────
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway, kruskal

# ── Scikit-learn ─────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    mean_squared_error, r2_score, mean_absolute_error
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('All libraries imported successfully.')

In [ ]:
# ── Load dataset ─────────────────────────────────────────────────────────────
df = pd.read_csv('Social_media_impact_on_life.csv')

print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
df.head()

In [ ]:
# ── Column data types & memory usage ─────────────────────────────────────────
df.info()

---
## 2. Data Cleaning & Quality Checks

In [ ]:
# ── Missing value summary ─────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]
print('Columns with missing values:')
print(missing_df)

In [ ]:
# ── Visualise missing values ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 3))
missing_pct[missing_pct > 0].plot(kind='barh', color='steelblue', ax=ax)
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Value Percentage by Column')
plt.tight_layout()
plt.show()

In [ ]:
# ── Fill missing GPA with median (only ~2–3% missing) ────────────────────────
# Median is preferred over mean to handle potential outlier GPAs
gpa_median = df['Academic_Performance_GPA'].median()
df['Academic_Performance_GPA'].fillna(gpa_median, inplace=True)

print(f'GPA median used for imputation: {gpa_median:.2f}')
print(f'Remaining missing values: {df.isnull().sum().sum()}')

In [ ]:
# ── Duplicate check ───────────────────────────────────────────────────────────
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates}')

# ── Drop Student_ID (not predictive) ─────────────────────────────────────────
df.drop(columns=['Student_ID'], inplace=True)

# ── Convert Late_Night_Usage to bool integer ──────────────────────────────────
df['Late_Night_Usage'] = df['Late_Night_Usage'].astype(int)

print('Cleaned dataset shape:', df.shape)
df.head(3)

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────────
df.describe().round(2)

In [ ]:
# ── Categorical column value counts ──────────────────────────────────────────
cat_cols = ['Gender', 'Academic_Level', 'Primary_Platform',
            'Device_Type', 'Social_Comparison_Frequency', 'Overall_Impact']

for col in cat_cols:
    print(f'\n{col}:')
    print(df[col].value_counts())

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── Distribution of Overall Impact (target variable) ─────────────────────────
impact_counts = df['Overall_Impact'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
impact_counts.plot(kind='bar', color=['#2ecc71', '#e67e22', '#e74c3c'], ax=axes[0], rot=0)
axes[0].set_title('Overall Impact — Count')
axes[0].set_ylabel('Number of Students')
axes[0].set_xlabel('')
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=10)

# Pie chart
axes[1].pie(impact_counts, labels=impact_counts.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e67e22', '#e74c3c'], startangle=90)
axes[1].set_title('Overall Impact — Distribution')

plt.suptitle('Distribution of Social Media Overall Impact on Students', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(impact_counts)

In [ ]:
# ── Age distribution ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(df['Age'], bins=15, kde=True, color='steelblue', ax=ax)
ax.set_title('Age Distribution of Students')
ax.set_xlabel('Age')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print(df['Age'].describe())

In [ ]:
# ── Gender distribution ───────────────────────────────────────────────────────
gender_counts = df['Gender'].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
gender_counts.plot(kind='bar', color=sns.color_palette('pastel'), ax=ax, rot=0)
ax.set_title('Gender Distribution')
ax.set_ylabel('Count')
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{int(bar.get_height()):,}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
# ── Daily Social Media Usage — histogram ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(df['Daily_Usage_Hours'], bins=25, kde=True, color='coral', ax=ax)
ax.axvline(df['Daily_Usage_Hours'].mean(), color='darkred', linestyle='--',
           label=f"Mean: {df['Daily_Usage_Hours'].mean():.2f} hrs")
ax.axvline(df['Daily_Usage_Hours'].median(), color='navy', linestyle=':',
           label=f"Median: {df['Daily_Usage_Hours'].median():.2f} hrs")
ax.legend()
ax.set_title('Distribution of Daily Social Media Usage (Hours)')
ax.set_xlabel('Daily Usage Hours')
plt.tight_layout()
plt.show()

print(df['Daily_Usage_Hours'].describe())

In [ ]:
# ── Platform popularity ───────────────────────────────────────────────────────
platform_counts = df['Primary_Platform'].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
platform_counts.plot(kind='bar', color=sns.color_palette('Set2', len(platform_counts)), ax=ax, rot=30)
ax.set_title('Most Used Social Media Platforms')
ax.set_ylabel('Number of Students')
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

print(platform_counts)

In [ ]:
# ── Sleep duration by Overall Impact ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
order = ['Beneficial', 'Neutral', 'Negative']
palette = {'Beneficial': '#2ecc71', 'Neutral': '#e67e22', 'Negative': '#e74c3c'}
sns.boxplot(data=df, x='Overall_Impact', y='Sleep_Duration_Hours',
            order=order, palette=palette, ax=ax)
ax.set_title('Sleep Duration by Overall Social Media Impact')
ax.set_xlabel('Overall Impact')
ax.set_ylabel('Sleep Duration (Hours)')
plt.tight_layout()
plt.show()

print(df.groupby('Overall_Impact')['Sleep_Duration_Hours'].describe().round(2))

In [ ]:
# ── Perceived Stress Score by Overall Impact ──────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
sns.violinplot(data=df, x='Overall_Impact', y='Perceived_Stress_Score',
               order=order, palette=palette, ax=ax, inner='quartile')
ax.set_title('Perceived Stress Score by Overall Impact')
ax.set_xlabel('Overall Impact')
ax.set_ylabel('Stress Score')
plt.tight_layout()
plt.show()

print(df.groupby('Overall_Impact')['Perceived_Stress_Score'].describe().round(2))

In [ ]:
# ── GPA distribution by Academic Level ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
level_order = ['High School', 'Undergraduate', 'Postgraduate']
sns.boxplot(data=df[df['Academic_Level'].isin(level_order)],
            x='Academic_Level', y='Academic_Performance_GPA',
            order=level_order, palette='Blues', ax=ax)
ax.set_title('GPA Distribution by Academic Level')
ax.set_xlabel('Academic Level')
ax.set_ylabel('GPA')
plt.tight_layout()
plt.show()

print(df.groupby('Academic_Level')['Academic_Performance_GPA'].describe().round(3))

In [ ]:
# ── Social Comparison Frequency vs Stress ────────────────────────────────────
freq_order = ['Never', 'Rarely', 'Sometimes', 'Frequently', 'Always']

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x='Social_Comparison_Frequency', y='Perceived_Stress_Score',
            order=freq_order, palette='RdYlGn_r', ax=ax)
ax.set_title('Perceived Stress Score by Social Comparison Frequency')
ax.set_xlabel('Social Comparison Frequency')
ax.set_ylabel('Stress Score')
plt.tight_layout()
plt.show()

print(df.groupby('Social_Comparison_Frequency')['Perceived_Stress_Score'].mean().reindex(freq_order).round(2))

In [ ]:
# ── Late-night usage — stress impact ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Stress
df.groupby('Late_Night_Usage')['Perceived_Stress_Score'].mean().plot(
    kind='bar', color=['#2ecc71', '#e74c3c'], ax=axes[0], rot=0)
axes[0].set_xticklabels(['No Late-Night Usage', 'Late-Night Usage'])
axes[0].set_title('Mean Stress Score by Late-Night Usage')
axes[0].set_ylabel('Mean Stress Score')

# Sleep
df.groupby('Late_Night_Usage')['Sleep_Duration_Hours'].mean().plot(
    kind='bar', color=['#2ecc71', '#e74c3c'], ax=axes[1], rot=0)
axes[1].set_xticklabels(['No Late-Night Usage', 'Late-Night Usage'])
axes[1].set_title('Mean Sleep Duration by Late-Night Usage')
axes[1].set_ylabel('Mean Sleep Hours')

plt.suptitle('Effect of Late-Night Social Media Usage', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(df.groupby('Late_Night_Usage')[['Perceived_Stress_Score', 'Sleep_Duration_Hours']].mean().round(2))

In [ ]:
# ── Daily usage hours by platform ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
platform_usage = df.groupby('Primary_Platform')['Daily_Usage_Hours'].mean().sort_values(ascending=False)
platform_usage.plot(kind='bar', color=sns.color_palette('tab10', len(platform_usage)), ax=ax, rot=30)
ax.set_title('Average Daily Usage Hours by Platform')
ax.set_ylabel('Mean Daily Hours')
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

---
## 4. Advanced Visualizations

In [ ]:
# ── Numeric correlation heatmap ───────────────────────────────────────────────
num_cols = ['Age', 'Daily_Usage_Hours', 'Weekend_Extra_Hours',
            'Sleep_Duration_Hours', 'Sleep_Quality_Score',
            'Perceived_Stress_Score', 'Mental_Health_Index',
            'Academic_Performance_GPA', 'Late_Night_Usage']

corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix — Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter: Daily Usage vs GPA, coloured by Overall Impact ──────────────────
fig, ax = plt.subplots(figsize=(9, 6))
color_map = {'Beneficial': '#2ecc71', 'Neutral': '#e67e22', 'Negative': '#e74c3c'}

for impact, grp in df.groupby('Overall_Impact'):
    ax.scatter(grp['Daily_Usage_Hours'], grp['Academic_Performance_GPA'],
               alpha=0.4, s=20, label=impact, color=color_map[impact])

# Regression line
m, b, r, p, se = stats.linregress(df['Daily_Usage_Hours'], df['Academic_Performance_GPA'])
x_line = np.linspace(df['Daily_Usage_Hours'].min(), df['Daily_Usage_Hours'].max(), 100)
ax.plot(x_line, m*x_line + b, 'k--', lw=2, label=f'Trend (r={r:.2f}, p={p:.3f})')

ax.set_title('Daily Social Media Usage vs GPA')
ax.set_xlabel('Daily Usage Hours')
ax.set_ylabel('GPA')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Pearson r = {r:.4f}, p-value = {p:.4f}')

In [ ]:
# ── Scatter: Stress Score vs Mental Health Index ──────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
for impact, grp in df.groupby('Overall_Impact'):
    ax.scatter(grp['Perceived_Stress_Score'], grp['Mental_Health_Index'],
               alpha=0.4, s=20, label=impact, color=color_map[impact])

m2, b2, r2, p2, _ = stats.linregress(df['Perceived_Stress_Score'], df['Mental_Health_Index'])
x2 = np.linspace(df['Perceived_Stress_Score'].min(), df['Perceived_Stress_Score'].max(), 100)
ax.plot(x2, m2*x2 + b2, 'k--', lw=2, label=f'Trend (r={r2:.2f})')

ax.set_title('Perceived Stress Score vs Mental Health Index')
ax.set_xlabel('Stress Score')
ax.set_ylabel('Mental Health Index')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Pearson r = {r2:.4f}, p-value = {p2:.4f}')

In [ ]:
# ── Stacked bar: Platform vs Overall Impact ───────────────────────────────────
platform_impact = (df.groupby(['Primary_Platform', 'Overall_Impact'])
                     .size().unstack(fill_value=0))
platform_impact_pct = platform_impact.div(platform_impact.sum(axis=1), axis=0) * 100

# Sort by Beneficial %
if 'Beneficial' in platform_impact_pct.columns:
    platform_impact_pct = platform_impact_pct.sort_values('Beneficial', ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
platform_impact_pct[['Beneficial', 'Neutral', 'Negative']].plot(
    kind='bar', stacked=True,
    color=['#2ecc71', '#e67e22', '#e74c3c'],
    ax=ax, rot=30)
ax.set_title('Overall Impact Distribution by Platform (%)')
ax.set_ylabel('Percentage of Students')
ax.set_xlabel('Platform')
ax.legend(title='Impact', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

print(platform_impact_pct.round(1))

In [ ]:
# ── Pairplot — key numeric features coloured by Overall Impact ────────────────
pairplot_cols = ['Daily_Usage_Hours', 'Sleep_Duration_Hours',
                 'Perceived_Stress_Score', 'Mental_Health_Index',
                 'Academic_Performance_GPA', 'Overall_Impact']

pair_df = df[pairplot_cols].copy()
g = sns.pairplot(pair_df, hue='Overall_Impact',
                 palette={'Beneficial': '#2ecc71', 'Neutral': '#e67e22', 'Negative': '#e74c3c'},
                 plot_kws={'alpha': 0.3, 's': 15},
                 diag_kind='kde')
g.fig.suptitle('Pairplot of Key Variables by Overall Impact', y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Heatmap: Mean GPA by Platform & Academic Level ───────────────────────────
pivot_gpa = df.pivot_table(values='Academic_Performance_GPA',
                            index='Academic_Level',
                            columns='Primary_Platform',
                            aggfunc='mean')

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot_gpa, annot=True, fmt='.2f', cmap='YlGnBu',
            linewidths=0.5, ax=ax, vmin=2.8, vmax=4.0)
ax.set_title('Mean GPA by Academic Level & Platform')
plt.tight_layout()
plt.show()

In [ ]:
# ── Daily usage vs Stress Score — faceted by Academic Level ──────────────────
g = sns.FacetGrid(df, col='Academic_Level', col_order=['High School', 'Undergraduate', 'Postgraduate'],
                  height=4, aspect=1.1, palette='Set1')
g.map_dataframe(sns.scatterplot, x='Daily_Usage_Hours', y='Perceived_Stress_Score',
                alpha=0.4, s=20, color='steelblue')
g.map_dataframe(sns.regplot, x='Daily_Usage_Hours', y='Perceived_Stress_Score',
                scatter=False, color='red', line_kws={'linewidth': 1.5})
g.set_titles(col_template='{col_name}')
g.set_axis_labels('Daily Usage Hours', 'Stress Score')
g.fig.suptitle('Daily Usage vs Stress by Academic Level', y=1.03, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Statistical Analysis

In [ ]:
# ── 5.1 Pearson Correlation — Daily Usage & key outcomes ─────────────────────
print('=== Pearson Correlations with Daily_Usage_Hours ===')
targets = ['Sleep_Duration_Hours', 'Sleep_Quality_Score', 'Perceived_Stress_Score',
           'Mental_Health_Index', 'Academic_Performance_GPA']

for col in targets:
    r, p = stats.pearsonr(df['Daily_Usage_Hours'], df[col])
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f'  Daily Usage vs {col:<30} r={r:+.4f}  p={p:.4f} {sig}')

In [ ]:
# ── 5.2 One-way ANOVA — Stress across Impact groups ──────────────────────────
groups = [grp['Perceived_Stress_Score'].values
          for _, grp in df.groupby('Overall_Impact')]

f_stat, p_anova = f_oneway(*groups)
print(f'ANOVA — Stress Score across Impact groups:')
print(f'  F-statistic = {f_stat:.4f}, p-value = {p_anova:.2e}')
print('  → Significant difference between groups' if p_anova < 0.05
      else '  → No significant difference')

# Group means for reference
print('\nGroup means:')
print(df.groupby('Overall_Impact')['Perceived_Stress_Score'].mean().round(2))

In [ ]:
# ── 5.3 Kruskal-Wallis test — GPA across impact groups (non-parametric) ──────
gpa_groups = [grp['Academic_Performance_GPA'].values
              for _, grp in df.groupby('Overall_Impact')]

h_stat, p_kruskal = kruskal(*gpa_groups)
print(f'Kruskal-Wallis — GPA across Impact groups:')
print(f'  H-statistic = {h_stat:.4f}, p-value = {p_kruskal:.2e}')
print('  → Significant difference' if p_kruskal < 0.05 else '  → No significant difference')

print('\nGPA medians by impact group:')
print(df.groupby('Overall_Impact')['Academic_Performance_GPA'].median().round(3))

In [ ]:
# ── 5.4 Chi-square test — Platform vs Overall Impact ─────────────────────────
contingency = pd.crosstab(df['Primary_Platform'], df['Overall_Impact'])
chi2, p_chi2, dof, expected = chi2_contingency(contingency)

print(f'Chi-Square Test — Platform vs Overall Impact:')
print(f'  χ² = {chi2:.4f}, dof = {dof}, p-value = {p_chi2:.4f}')
print('  → Significant association' if p_chi2 < 0.05
      else '  → No significant association')

print('\nContingency table (counts):')
print(contingency)

In [ ]:
# ── 5.5 Independent t-test — Late-night usage effect on GPA ──────────────────
late = df[df['Late_Night_Usage'] == 1]['Academic_Performance_GPA']
not_late = df[df['Late_Night_Usage'] == 0]['Academic_Performance_GPA']

t_stat, p_t = stats.ttest_ind(late, not_late)
print(f'T-test — GPA: Late-night users vs Non-late-night users')
print(f'  Late-night mean GPA    = {late.mean():.4f}')
print(f'  Non-late-night mean GPA = {not_late.mean():.4f}')
print(f'  t-statistic = {t_stat:.4f}, p-value = {p_t:.4f}')
print('  → Significant difference' if p_t < 0.05 else '  → No significant difference')

In [ ]:
# ── 5.6 Spearman correlation matrix — rank-based ──────────────────────────────
spearman_corr = df[num_cols].corr(method='spearman')

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(spearman_corr, dtype=bool))
sns.heatmap(spearman_corr, mask=mask, annot=True, fmt='.2f',
            cmap='PiYG', center=0, linewidths=0.5, ax=ax)
ax.set_title('Spearman Rank-Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Machine Learning — Classification (Predict Overall Impact)

In [ ]:
# ── Feature Engineering — encode categorical variables ────────────────────────
df_ml = df.copy()

# Ordinal encode Social_Comparison_Frequency
freq_map = {'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Frequently': 3, 'Always': 4}
df_ml['Social_Comparison_Freq_Enc'] = df_ml['Social_Comparison_Frequency'].map(freq_map)

# Ordinal encode Academic_Level
level_map = {'High School': 0, 'Undergraduate': 1, 'Postgraduate': 2}
df_ml['Academic_Level_Enc'] = df_ml['Academic_Level'].map(level_map)

# Label encode Gender
le_gender = LabelEncoder()
df_ml['Gender_Enc'] = le_gender.fit_transform(df_ml['Gender'])

# One-hot encode Platform and Device
df_ml = pd.get_dummies(df_ml, columns=['Primary_Platform', 'Device_Type'], drop_first=True)

# Encode target
impact_map = {'Beneficial': 0, 'Neutral': 1, 'Negative': 2}
df_ml['Impact_Label'] = df_ml['Overall_Impact'].map(impact_map)

print('Encoded DataFrame shape:', df_ml.shape)

In [ ]:
# ── Select features for classification ───────────────────────────────────────
drop_cols = ['Gender', 'Academic_Level', 'Social_Comparison_Frequency', 'Overall_Impact', 'Impact_Label']
feature_cols = [c for c in df_ml.columns if c not in drop_cols]

X = df_ml[feature_cols]
y = df_ml['Impact_Label']

# Train-test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training samples  : {X_train.shape[0]}')
print(f'Test samples      : {X_test.shape[0]}')
print(f'Number of features: {X_train.shape[1]}')

In [ ]:
# ── Train & compare classifiers ───────────────────────────────────────────────
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, C=1.0),
    'Decision Tree':       DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=42)
}

results = {}
for name, clf in classifiers.items():
    # Use scaled data for Logistic Regression, raw for tree-based models
    if 'Logistic' in name:
        clf.fit(X_train_sc, y_train)
        y_pred = clf.predict(X_test_sc)
    else:
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = {'model': clf, 'preds': y_pred, 'accuracy': acc}
    print(f'{name:<25} Accuracy: {acc:.4f}')

In [ ]:
# ── Accuracy comparison bar chart ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
model_names  = list(results.keys())
accuracies   = [results[n]['accuracy'] for n in model_names]
colors = sns.color_palette('Set2', len(model_names))
bars = ax.bar(model_names, accuracies, color=colors)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy')
ax.set_title('Classifier Accuracy Comparison — Overall Impact Prediction')
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.3f}', ha='center', va='bottom', fontsize=10)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# ── Best model — detailed evaluation ─────────────────────────────────────────
best_name = max(results, key=lambda n: results[n]['accuracy'])
best_preds = results[best_name]['preds']

print(f'Best classifier: {best_name}')
print(f'Accuracy: {results[best_name]["accuracy"]:.4f}\n')

class_names = ['Beneficial', 'Neutral', 'Negative']
print(classification_report(y_test, best_preds, target_names=class_names))

In [ ]:
# ── Confusion matrix — best classifier ───────────────────────────────────────
cm = confusion_matrix(y_test, best_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title(f'Confusion Matrix — {best_name}')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Importance — Random Forest ───────────────────────────────────────
rf_model = results['Random Forest']['model']
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
importances.head(15).plot(kind='barh', ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_title('Top 15 Feature Importances (Random Forest — Classification)')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('\nTop 10 most important features:')
print(importances.head(10).round(4))

In [ ]:
# ── 5-fold cross-validation — Random Forest ───────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)

print(f'5-Fold CV Accuracy scores: {cv_scores.round(4)}')
print(f'Mean CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

---
## 7. Machine Learning — Regression (Predict GPA)

In [ ]:
# ── GPA regression setup ──────────────────────────────────────────────────────
y_gpa = df_ml['Academic_Performance_GPA']

# Re-use the same feature set (excluding GPA from features)
feature_cols_reg = [c for c in feature_cols if c != 'Academic_Performance_GPA']
X_reg = df_ml[feature_cols_reg]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_reg, y_gpa, test_size=0.2, random_state=42)

scaler_r = StandardScaler()
X_tr_sc = scaler_r.fit_transform(X_tr)
X_te_sc  = scaler_r.transform(X_te)

print(f'Regression — Train: {X_tr.shape[0]}  Test: {X_te.shape[0]}')

In [ ]:
# ── Train & evaluate regression models ───────────────────────────────────────
regressors = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=1.0),
    'Random Forest Reg': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
}

reg_results = {}
for name, reg in regressors.items():
    if 'Forest' not in name:
        reg.fit(X_tr_sc, y_tr)
        y_pred_r = reg.predict(X_te_sc)
    else:
        reg.fit(X_tr, y_tr)
        y_pred_r = reg.predict(X_te)

    rmse = np.sqrt(mean_squared_error(y_te, y_pred_r))
    mae  = mean_absolute_error(y_te, y_pred_r)
    r2   = r2_score(y_te, y_pred_r)
    reg_results[name] = {'model': reg, 'preds': y_pred_r, 'rmse': rmse, 'mae': mae, 'r2': r2}
    print(f'{name:<22} RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}')

In [ ]:
# ── Actual vs Predicted GPA — best regressor ─────────────────────────────────
best_reg = max(reg_results, key=lambda n: reg_results[n]['r2'])
y_pred_best = reg_results[best_reg]['preds']

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_te, y_pred_best, alpha=0.3, s=15, color='steelblue')
lims = [min(y_te.min(), y_pred_best.min()), max(y_te.max(), y_pred_best.max())]
ax.plot(lims, lims, 'r--', lw=2, label='Perfect Fit')
ax.set_xlabel('Actual GPA')
ax.set_ylabel('Predicted GPA')
ax.set_title(f'Actual vs Predicted GPA — {best_reg}\nR² = {reg_results[best_reg]["r2"]:.4f}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Residuals plot ────────────────────────────────────────────────────────────
residuals = y_te.values - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Residuals vs Fitted
axes[0].scatter(y_pred_best, residuals, alpha=0.3, s=15, color='coral')
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set_xlabel('Predicted GPA')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Fitted')

# Residual distribution
axes[1].hist(residuals, bins=30, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Residual')
axes[1].set_title('Residual Distribution')

plt.suptitle(f'Residual Analysis — {best_reg}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Importance — Random Forest Regressor ──────────────────────────────
rf_reg = reg_results['Random Forest Reg']['model']
reg_importances = pd.Series(rf_reg.feature_importances_,
                             index=feature_cols_reg).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
reg_importances.head(15).plot(kind='barh', ax=ax, color='darkorange')
ax.invert_yaxis()
ax.set_title('Top 15 Feature Importances (Random Forest — GPA Regression)')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Top 10 predictors of GPA:')
print(reg_importances.head(10).round(4))

---
## 8. Key Findings & Conclusions

In [ ]:
# ── Final summary statistics ──────────────────────────────────────────────────
print('=' * 65)
print('       IMPACT OF SOCIAL MEDIA ON STUDENT LIFE — SUMMARY')
print('=' * 65)

print(f"\n📊 Dataset: {len(df):,} students across {df['Academic_Level'].nunique()} academic levels")
print(f"    Age range: {df['Age'].min()}–{df['Age'].max()} | Mean age: {df['Age'].mean():.1f}")

print(f"\n📱 Social Media Usage:")
print(f"    Mean daily usage: {df['Daily_Usage_Hours'].mean():.2f} hrs/day")
print(f"    Max daily usage : {df['Daily_Usage_Hours'].max()} hrs/day")
print(f"    Most popular    : {df['Primary_Platform'].mode()[0]}")
print(f"    Late-night users: {df['Late_Night_Usage'].mean()*100:.1f}%")

print(f"\n😴 Sleep:")
print(f"    Mean sleep duration: {df['Sleep_Duration_Hours'].mean():.2f} hrs")
r_sleep, p_sleep = stats.pearsonr(df['Daily_Usage_Hours'], df['Sleep_Duration_Hours'])
print(f"    Correlation (usage vs sleep): r={r_sleep:.3f} (p={p_sleep:.4f})")

print(f"\n🧠 Mental Health:")
r_stress, _ = stats.pearsonr(df['Daily_Usage_Hours'], df['Perceived_Stress_Score'])
print(f"    Correlation (usage vs stress): r={r_stress:.3f}")
r_mh, _ = stats.pearsonr(df['Perceived_Stress_Score'], df['Mental_Health_Index'])
print(f"    Correlation (stress vs mental health): r={r_mh:.3f}")

print(f"\n📚 Academic Performance:")
r_gpa, p_gpa = stats.pearsonr(df['Daily_Usage_Hours'], df['Academic_Performance_GPA'])
print(f"    Correlation (usage vs GPA): r={r_gpa:.3f} (p={p_gpa:.4f})")
print(f"    Mean GPA: {df['Academic_Performance_GPA'].mean():.3f}")

print(f"\n🤖 ML Results:")
print(f"    Best Classifier  : {best_name}")
print(f"    Classification Accuracy: {results[best_name]['accuracy']:.4f}")
print(f"    Best GPA Regressor: {best_reg}")
print(f"    Regression R²    : {reg_results[best_reg]['r2']:.4f}")
print(f"    Regression RMSE  : {reg_results[best_reg]['rmse']:.4f}")

print('\n' + '=' * 65)
print('Key Insights:')
print('  1. Higher daily usage significantly correlates with lower sleep')
print('     duration and elevated stress (both p < 0.001).')
print('  2. Students using social media late at night report higher stress')
print('     and shorter sleep compared to those who do not.')
print('  3. Frequent social comparison is the strongest single predictor')
print('     of perceived stress.')
print('  4. GPA shows a weak negative correlation with usage, suggesting')
print('     usage alone is not the primary academic driver.')
print('  5. The majority (≈83%) of students report a Beneficial overall')
print('     impact — few report Negative outcomes.')
print('=' * 65)